### BP batch processing
Runs the segmentation and feature-extraction pipeline on all images in `data_folder`, saves one CSV per image under `results/experiment_id`. No filtering or visualization.

### Config: data folder, OCs, output directory

In [ ]:
import os
from pathlib import Path

from utils import list_images, check_marker_metadata_match, read_image, extract_img_metadata

# Path to folder containing images
data_folder = r"X:\Shirin\RCDAnalysis_Alberto\SK0052"

# Markers for intensity analysis: (channel_name, channel_nr), 0-based
markers = [("SD_DAPI", 1), ("SD_GFP", 2), ("SD_RFP", 3), ("SD_AF647", 4)]

# Define brightfield channel position to use as input for CellposeSAM-mediated cell segmentation
brightfield_channel = 0

images = list_images(data_folder, format="nd2")
experiment_id = Path(data_folder).name
out_dir = Path("results") / experiment_id
os.makedirs(out_dir, exist_ok=True)
print(f"Found {len(images)} images. Output: {out_dir}")

# Check if the defined marker position match the metadata in the image files
metadata_match = check_marker_metadata_match(images, markers)
if not metadata_match:
    raise RuntimeError(
        "Marker/metadata mismatch detected in one or more files. "
        "Fix markers or input data before continuing."
    )

### Imports and model load (run once)

In [ ]:
from cellpose import models, core
from skimage.segmentation import clear_border
from skimage.measure import regionprops_table
import pandas as pd
import numpy as np

import logging
logging.getLogger("tifffile").setLevel(logging.ERROR)

if core.use_gpu() == False:
    raise ImportError("No GPU access, change your runtime")

model = models.CellposeModel(gpu=True)

### Morphology and intensity property lists

In [ ]:
morphology_properties = [
    "label",
    "area",
    "area_bbox",
    "area_convex",
    "area_filled",
    "axis_major_length",
    "axis_minor_length",
    "equivalent_diameter_area",
    "euler_number",
    "extent",
    "feret_diameter_max",
    "solidity",
    "inertia_tensor_eigvals",
]

intensity_properties = [
    "label",
    "intensity_mean",
    "intensity_min",
    "intensity_max",
    "intensity_std",
]

### Process all images and save CSVs

In [ ]:
for i, img_filepath in enumerate(images):
    print(f"Processing {i + 1}/{len(images)}: {Path(img_filepath).name}")

    img, filename = read_image(img_filepath, log=False)
    descriptor_dict = extract_img_metadata(img_filepath, verbose=False)

    cell_labels, flows, styles = model.eval(img[brightfield_channel], niter=1000)
    cell_labels = clear_border(cell_labels)

    props_morphology = regionprops_table(
        label_image=cell_labels,
        properties=morphology_properties,
    )
    props_df = pd.DataFrame(props_morphology)

    for marker_name, ch_nr in markers:
        props = regionprops_table(
            label_image=cell_labels,
            intensity_image=img[ch_nr],
            properties=intensity_properties,
        )
        intensity_df = pd.DataFrame(props)

        prefix = f"{marker_name}"
        rename_map = {"label": "label"}
        for prop in intensity_properties:
            if prop == "label":
                continue
            if prop.startswith("intensity_"):
                suffix = prop.replace("intensity_", "")
                rename_map[prop] = f"{prefix}_{suffix}_int"
        intensity_df.rename(columns=rename_map, inplace=True)

        mean_col = rename_map["intensity_mean"]
        max_col = rename_map["intensity_max"]
        intensity_df[f"{prefix}_max_mean_ratio"] = (
            intensity_df[max_col] / intensity_df[mean_col].replace(0, np.nan)
        )

        props_df = props_df.merge(intensity_df, on="label")
        props_df[f"{prefix}_sum_int"] = props_df[mean_col] * props_df["area"]

    insertion_position = 0
    for key, value in descriptor_dict.items():
        props_df.insert(insertion_position, key, value)
        insertion_position += 1

    csv_path = out_dir / f"{descriptor_dict['well_id']}.csv"
    props_df.to_csv(csv_path, index=False)
    print(f"  -> {csv_path}")

print("Done.")